In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import chess
import chess.pgn

# ==========================================
# 1. FINAL VOCABULARY & CONFIG
# ==========================================
VOCAB = {
    'EMPTY': 0,
    'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,    # White Pieces
    'p': 7, 'n': 8, 'b': 9, 'r': 10, 'q': 11, 'k': 12,  # Black Pieces
    '[W_WIN]': 13, '[B_WIN]': 14, '[DRAW]': 15,        # Win Conditions
    '[INIT]': 16, '[W_MOVE]': 17, '[B_MOVE]': 18,      # Turn Context
    '[W_KSC]': 19, '[W_QSC]': 20, '[B_KSC]': 21, '[B_QSC]': 22, # Castle Rights
    '[NO_CASTLE]': 23,                                 # Smart fix
    '[EP_A]': 24, '[EP_B]': 25, '[EP_C]': 26, '[EP_D]': 27, 
    '[EP_E]': 28, '[EP_F]': 29, '[EP_G]': 30, '[EP_H]': 31, '[EP_NONE]': 32,
    '[ELO_0]': 33, '[ELO_1]': 34, '[ELO_2]': 35, '[ELO_3]': 36, '[ELO_4]': 37,
    '[ELO_5]': 38, '[ELO_6]': 39, '[ELO_7]': 40, '[ELO_8]': 41, '[ELO_9]': 42,
    '[PAD]': 43, '[MASK]': 44
}

# ==========================================
# 2. VIT-HYBRID MODEL ARCHITECTURE
# ==========================================
class ViTChessHybrid(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = len(VOCAB)

        # A. Metadata Embedding (9 Tokens: Goal, Elo x2, Turn, Castle x4, EP)
        self.meta_embedding = nn.Embedding(self.vocab_size, d_model)
        self.meta_pos_encoding = nn.Parameter(torch.randn(1, 9, d_model))

        # B. Board Embedding (64 Squares)
        self.board_embedding = nn.Embedding(self.vocab_size, d_model)
        # 2D Positional Encoding: We learn a specific vector for each (x,y) coordinate
        self.pos_embed_2d = nn.Parameter(torch.randn(1, 64, d_model))

        # C. Transformer Backbone
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dropout=dropout
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # D. Dual-Heads
        self.policy_head = nn.Linear(d_model, self.vocab_size)
        self.value_head = nn.Sequential(
            nn.Linear(d_model * 73, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh() # Score from -1.0 to 1.0
        )

    def forward(self, x):
        # x input: [batch, 73]
        batch_size = x.size(0)

        # 1. Process Metadata (Indices 0-8)
        meta_tokens = x[:, :9]
        meta_emb = self.meta_embedding(meta_tokens) + self.meta_pos_encoding

        # 2. Process Board (Indices 9-72)
        board_tokens = x[:, 9:]
        board_emb = self.board_embedding(board_tokens) + self.pos_embed_2d

        # 3. Hybrid Fusion: Concatenate metadata + board patches
        # sequence shape: [batch, 73, 256]
        full_seq = torch.cat([meta_emb, board_emb], dim=1)

        # 4. Global Attention (Self-Attention across all 73 tokens)
        latent = self.transformer(full_seq)

        # 5. Extract Policy (Next state for the 64 board squares)
        board_latent = latent[:, 9:, :] 
        policy_logits = self.policy_head(board_latent) # [batch, 64, 45]

        # 6. Extract Value (Global Evaluation)
        flat_latent = latent.reshape(batch_size, -1)
        value_score = self.value_head(flat_latent) # [batch, 1]

        return policy_logits, value_score

class ChessLoss(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.policy_criterion = nn.CrossEntropyLoss()
        self.value_criterion = nn.MSELoss()

    def forward(self, pred_policy, target_policy, pred_value, target_value):
        p_loss = self.policy_criterion(
            pred_policy.view(-1, pred_policy.size(-1)), 
            target_policy.view(-1)
        )
        v_loss = self.value_criterion(pred_value.squeeze(), target_value)
        return p_loss + v_loss


In [ ]:
from torch.utils.data import IterableDataset, DataLoader

# ==========================================
# 3. DATA PREPROCESSING (IterableDataset to prevent OOM)
# ==========================================

def encode_board_for_vit(board, win_goal_token, white_elo_token, black_elo_token):
    """
    Converts a chess.Board into the specific 73-token sequence 
    required for the ViT Hybrid model.
    """
    # 9-token Metadata Header
    seq = [win_goal_token, white_elo_token, black_elo_token]
    
    # Turn/Init
    if board.fullmove_number == 1 and board.turn == chess.WHITE:
        seq.append(VOCAB['[INIT]'])
    else:
        seq.append(VOCAB['[W_MOVE]'] if board.turn == chess.WHITE else VOCAB['[B_MOVE]'])
        
    # Castling Rights (with NO_CASTLE fix)
    NC = VOCAB['[NO_CASTLE]']
    seq.append(VOCAB['[W_KSC]'] if board.has_kingside_castling_rights(chess.WHITE) else NC)
    seq.append(VOCAB['[W_QSC]'] if board.has_queenside_castling_rights(chess.WHITE) else NC)
    seq.append(VOCAB['[B_KSC]'] if board.has_kingside_castling_rights(chess.BLACK) else NC)
    seq.append(VOCAB['[B_QSC]'] if board.has_queenside_castling_rights(chess.BLACK) else NC)
    
    # En Passant
    ep = board.ep_square
    seq.append(VOCAB[f'[EP_{chess.square_name(ep)[0].upper()}]'] if ep else VOCAB['[EP_NONE]'])
    
    # 64-token Board Body
    for s in chess.SQUARES:
        p = board.piece_at(s)
        seq.append(VOCAB[p.symbol()] if p else VOCAB['EMPTY'])
        
    # Return flat list for dataset batching
    return seq

class ChessDataset(IterableDataset):
    def __init__(self, pgn_files):
        if isinstance(pgn_files, str):
            self.pgn_files = [pgn_files]
        else:
            self.pgn_files = pgn_files

    def _get_elo_token(self, elo_str):
        try:
            elo = int(elo_str)
            if elo < 1000: return VOCAB['[ELO_0]']
            if elo >= 2600: return VOCAB['[ELO_9]']
            return VOCAB[f'[ELO_{(elo - 1000) // 200 + 1}]']
        except: 
            return VOCAB['[ELO_4]'] # Default jump

    def __iter__(self):
        for pgn_path in self.pgn_files:
            with open(pgn_path) as pgn:
                while True:
                    game = chess.pgn.read_game(pgn)
                    if not game: break
                    
                    res = game.headers.get("Result")
                    win_token = VOCAB['[W_WIN]'] if res == "1-0" else VOCAB['[B_WIN]'] if res == "0-1" else VOCAB['[DRAW]']
                    
                    # Extract ELOs
                    w_elo = self._get_elo_token(game.headers.get("WhiteElo", "?"))
                    b_elo = self._get_elo_token(game.headers.get("BlackElo", "?"))
                    
                    board = game.board()
                    for move in game.mainline_moves():
                        # Input context: Before move
                        input_seq = encode_board_for_vit(board, win_token, w_elo, b_elo)
                        
                        # Value Target: from the perspective of the player WHOSE TURN IT IS in the input state
                        if res == "1-0":
                            v_target = 1.0 if board.turn == chess.WHITE else -1.0
                        elif res == "0-1":
                            v_target = -1.0 if board.turn == chess.WHITE else 1.0
                        else:
                            v_target = 0.0
                            
                        # Policy Target: Board after move
                        board.push(move)
                        p_target = [VOCAB[board.piece_at(s).symbol()] if board.piece_at(s) else VOCAB['EMPTY'] for s in chess.SQUARES]
                        
                        yield (
                            torch.tensor(input_seq, dtype=torch.long), 
                            torch.tensor(p_target, dtype=torch.long), 
                            torch.tensor(v_target, dtype=torch.float32)
                        )


In [ ]:
import os
pgn_files = [f"ChessData/{f}" for f in os.listdir("ChessData") if f.endswith(".pgn")]
dataset = ChessDataset(pgn_files)
dataloader = DataLoader(dataset, batch_size=256)


In [ ]:
import time
import math
import os
from datetime import datetime

# ==========================================
# 4. TRAINING LOOP
# ==========================================
def train(epochs=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_dir = f"checkpoints/run_{run_timestamp}"
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Checkpoints will be saved to: {checkpoint_dir}")
    
    best_checkpoint_path = f"{checkpoint_dir}/chess_vit_best.pt"
    latest_checkpoint_path = f"{checkpoint_dir}/chess_vit_latest.pt"
    
    model = ViTChessHybrid(d_model=256, nhead=8, num_layers=8, dropout=0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = ChessLoss(len(VOCAB)).to(device)
    
    model.train()
    best_loss = math.inf
    
    for epoch in range(epochs):
        print(f"\n
Starting Epoch {epoch+1}/{epochs}")
        running_loss = 0.0
        start_time = time.time()
        
        for batch_idx, (inputs, targets_policy, targets_value) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets_policy = targets_policy.to(device)
            targets_value = targets_value.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            policy_pred, value_pred = model(inputs)
            
            # Multi-task loss
            loss = criterion(policy_pred, targets_policy, value_pred, targets_value)
            loss.backward()
            
            # Clip Gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            current_loss = loss.item()
            running_loss += current_loss
            
            if (batch_idx + 1) % 100 == 0:
                avg_loss = running_loss / 100
                elapsed = time.time() - start_time
                print(f"Batch {batch_idx+1:05d} | Avg Loss (last 100): {avg_loss:.4f} | Time: {elapsed:.2f}s")
                
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    torch.save(model.state_dict(), best_checkpoint_path)
                    print(f"  --> New best model saved with loss: {best_loss:.4f}")
                    
                running_loss = 0.0
                start_time = time.time()
                
            if (batch_idx + 1) % 10000 == 0:
                 torch.save(model.state_dict(), latest_checkpoint_path)

        torch.save(model.state_dict(), latest_checkpoint_path)
        print(f"Epoch {epoch+1} completed. Best Multi-task Loss: {best_loss:.4f}. Latest model saved.")

# train(epochs=2)


In [ ]:
# To start training, simply call the function:
train(epochs=3)